In [2]:
import pandas as pd
import numpy as np
import sqlite3

print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)

Pandas Version: 3.1.1
NumPy Version: 2.5.4


In [3]:
conn = sqlite3.connect("reel_insights.db")

print("Connected to Reel Insights database successfully!")

Connected to Reel Insights database successfully!


In [5]:
movies = pd.read_sql_query("SELECT * FROM movies", conn)
genres = pd.read_sql_query("SELECT * FROM genres", conn)
movie_genres = pd.read_sql_query("SELECT * FROM movie_genres", conn)
cast = pd.read_sql_query('SELECT * FROM "cast"', conn)
crew = pd.read_sql_query("SELECT * FROM crew", conn)
movie_keywords = pd.read_sql_query("SELECT * FROM movie_keywords", conn)

print("Movies:", movies.shape)
print("Genres:", genres.shape)
print("Movie Genres:", movie_genres.shape)
print("Cast:", cast.shape)
print("Crew:", crew.shape)
print("Movie Keywords:", movie_keywords.shape)

Movies: (2503, 18)
Genres: (19, 2)
Movie Genres: (6878, 2)
Cast: (24902, 6)
Crew: (7289, 6)
Movie Keywords: (41681, 3)


In [6]:
unique_counts = pd.DataFrame({
    "Category": [
        "Movies",
        "Genres",
        "Cast Members",
        "Crew Members"
    ],
    "Unique_Count": [
        movies["movie_id"].nunique(),
        genres["genre_id"].nunique(),
        cast["person_id"].nunique(),
        crew["person_id"].nunique()
    ]
})

display(unique_counts)

,Category,Unique_Count
0,Movies,2503
1,Genres,19
2,Cast Members,11300
3,Crew Members,2848


In [7]:
financial_quality = pd.DataFrame({
    "Metric": ["Budget", "Revenue"],
    "Missing_Values": [
        movies["budget_clean"].isna().sum(),
        movies["revenue_clean"].isna().sum()
    ],
    "Zero_Values": [
        movies["budget"].eq(0).sum(),
        movies["revenue"].eq(0).sum()
    ]
})

display(financial_quality)

,Metric,Missing_Values,Zero_Values
0,Budget,90,90
1,Revenue,102,102


In [8]:
financial_movies = movies.dropna(
    subset=["budget_clean", "revenue_clean"]
).copy()

financial_movies["profit"] = (
    financial_movies["revenue_clean"] -
    financial_movies["budget_clean"]
)

financial_movies["roi"] = (
    financial_movies["profit"] /
    financial_movies["budget_clean"]
) * 100

display(
    financial_movies[
        ["title", "budget_clean", "revenue_clean", "profit", "roi"]
    ].head(10)
)

print("Movies with valid financial data:", len(financial_movies))

,title,budget_clean,revenue_clean,profit,roi
0,Four Rooms,4000000.0,4257354.0,257354.0,6.433850
1,Star Wars,11000000.0,775398007.0,764398007.0,6949.072791
2,Finding Nemo,94000000.0,940335536.0,846335536.0,900.356953
3,Forrest Gump,55000000.0,677387716.0,622387716.0,1131.614029
4,American Beauty,15000000.0,356296601.0,341296601.0,2275.310673
5,Citizen Kane,839727.0,23218000.0,22378273.0,2664.946227
6,The Fifth Element,90000000.0,263920180.0,173920180.0,193.244644
7,Metropolis,5300000.0,1350322.0,-3949678.0,-74.522226
8,Pirates of the Caribbean: The Curse of the Bla...,140000000.0,655011224.0,515011224.0,367.865160
9,Kill Bill: Vol. 1,30000000.0,180906076.0,150906076.0,503.020253


Movies with valid financial data: 2370


In [9]:
top_roi_movies = financial_movies[
    financial_movies["budget_clean"] >= 1000000
].copy()

top_roi_movies = top_roi_movies.sort_values(
    by="roi",
    ascending=False
).head(15)

display(
    top_roi_movies[
        ["title", "budget_clean", "revenue_clean", "profit", "roi"]
    ]
)

,title,budget_clean,revenue_clean,profit,roi
2191,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,12400.282100
152,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,12324.256142
1211,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,12127.242857
412,Rocky,1000000.0,117253345.0,116253345.0,11625.334500
300,Gone with the Wind,4000000.0,402352579.0,398352579.0,9958.814475
794,The Jungle Book,4000000.0,378000000.0,374000000.0,9350.000000
1001,Cinderella,2900000.0,263600000.0,260700000.0,8989.655172
78,Saw,1200000.0,104045735.0,102845735.0,8570.477917
1055,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,8316.666667
221,E.T. the Extra-Terrestrial,10500000.0,797307407.0,786807407.0,7493.403876


In [10]:
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

movies["release_year"] = movies["release_date"].dt.year

movies_per_year = (
    movies.groupby("release_year")["movie_id"]
    .nunique()
    .reset_index(name="total_movies")
    .sort_values("release_year")
)

display(movies_per_year)

peak_year = movies_per_year.loc[
    movies_per_year["total_movies"].idxmax()
]

print("Peak Release Year:", int(peak_year["release_year"]))
print("Number of Movies:", int(peak_year["total_movies"]))

,release_year,total_movies
0,1921,1
1,1922,1
2,1927,1
3,1931,2
4,1936,1
...,...,...
84,2022,65
85,2023,62
86,2024,43
87,2025,26


Peak Release Year: 2016
Number of Movies: 121


In [11]:
runtime = movies["runtime"].dropna()

Q1 = runtime.quantile(0.25)
Q3 = runtime.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

runtime_outliers = movies[
    (movies["runtime"] < lower_bound) |
    (movies["runtime"] > upper_bound)
].copy()

runtime_outliers = runtime_outliers.sort_values(
    by="runtime",
    ascending=False
)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Total Runtime Outliers:", len(runtime_outliers))

display(
    runtime_outliers[
        ["title", "runtime", "release_year"]
    ].head(20)
)

Q1: 99.0
Q3: 126.0
IQR: 27.0
Lower Bound: 58.5
Upper Bound: 166.5
Total Runtime Outliers: 60


,title,runtime,release_year
2409,Zack Snyder's Justice League,242,2021
300,Gone with the Wind,238,1939
124,Once Upon a Time in America,229,1984
374,Lawrence of Arabia,228,1962
260,Ben-Hur,212,1959
2015,The Irishman,209,2019
136,Seven Samurai,207,1954
2150,Killers of the Flower Moon,206,2023
100,The Godfather Part II,202,1974
58,The Lord of the Rings: The Return of the King,201,2003


In [12]:
# Select one primary genre for each movie
primary_genres = (
    movie_genres.sort_values(["movie_id", "genre_id"])
    .drop_duplicates(subset=["movie_id"], keep="first")
)

# Add genre names
primary_genres = primary_genres.merge(
    genres,
    on="genre_id",
    how="left"
)

# Add movie popularity
primary_genres = primary_genres.merge(
    movies[["movie_id", "title", "popularity"]],
    on="movie_id",
    how="left"
)

# Calculate average popularity for each genre
genre_popularity = (
    primary_genres.groupby("genre_name")
    .agg(
        total_movies=("movie_id", "nunique"),
        average_popularity=("popularity", "mean")
    )
    .reset_index()
    .sort_values("average_popularity", ascending=False)
)

genre_popularity["average_popularity"] = (
    genre_popularity["average_popularity"].round(2)
)

display(genre_popularity)

,genre_name,total_movies,average_popularity
11,Thriller,62,18.85
1,Adventure,674,17.37
2,Animation,55,15.23
9,Horror,252,14.60
4,Crime,1,14.59
13,Western,7,13.71
0,Action,254,13.28
10,Science Fiction,3,12.60
7,Fantasy,200,12.42
6,Drama,779,11.15


In [13]:
# Join cast details with movie ratings
actor_movies = cast.merge(
    movies[["movie_id", "vote_average"]],
    on="movie_id",
    how="inner"
)

# Keep only one record per actor per movie
actor_movies = actor_movies[
    ["person_id", "actor_name", "movie_id", "vote_average"]
].drop_duplicates(
    subset=["person_id", "movie_id"]
)

# Calculate total movies and average rating for each actor
top_actors = (
    actor_movies.groupby(["person_id", "actor_name"])
    .agg(
        total_movies=("movie_id", "nunique"),
        average_rating=("vote_average", "mean")
    )
    .reset_index()
    .sort_values(
        ["total_movies", "actor_name"],
        ascending=[False, True]
    )
    .head(10)
)

top_actors["average_rating"] = (
    top_actors["average_rating"].round(2)
)

display(top_actors)

,person_id,actor_name,total_movies,average_rating
679,2231,Samuel L. Jackson,39,6.94
106,287,Brad Pitt,38,7.26
127,380,Robert De Niro,38,7.23
37,85,Johnny Depp,37,6.86
13,31,Tom Hanks,33,7.29
2674,13240,Mark Wahlberg,32,6.63
413,1245,Scarlett Johansson,32,7.06
1272,5293,Willem Dafoe,32,7.11
64,192,Morgan Freeman,31,6.96
161,500,Tom Cruise,31,7.04


In [14]:
# Find movies with duplicate titles and release years
duplicate_movies = movies[
    movies.duplicated(
        subset=["title", "release_year"],
        keep=False
    )
].copy()

# Sort the results
duplicate_movies = duplicate_movies.sort_values(
    by=["title", "release_year"]
)

print("Total duplicate title/year records:", len(duplicate_movies))

display(
    duplicate_movies[
        ["movie_id", "title", "release_year", "original_language"]
    ]
)

Total duplicate title/year records: 0


,movie_id,title,release_year,original_language


In [15]:
# Keep valid popularity and vote count values
anomaly_movies = movies[
    (movies["popularity"] >= 0) &
    (movies["vote_count"] >= 0)
].copy()

# Apply log transformation to vote count
anomaly_movies["log_vote_count"] = np.log1p(
    anomaly_movies["vote_count"]
)

# Calculate Z-scores
anomaly_movies["popularity_zscore"] = (
    anomaly_movies["popularity"] -
    anomaly_movies["popularity"].mean()
) / anomaly_movies["popularity"].std()

anomaly_movies["vote_count_zscore"] = (
    anomaly_movies["log_vote_count"] -
    anomaly_movies["log_vote_count"].mean()
) / anomaly_movies["log_vote_count"].std()

# Identify anomalies using a 3 standard deviation threshold
anomaly_movies["is_anomaly"] = (
    (anomaly_movies["popularity_zscore"].abs() > 3) |
    (anomaly_movies["vote_count_zscore"].abs() > 3)
)

# Filter anomaly records
anomalies = anomaly_movies[
    anomaly_movies["is_anomaly"]
].copy()

# Sort by popularity Z-score
anomalies = anomalies.sort_values(
    by="popularity_zscore",
    ascending=False
)

print("Total anomaly movies:", len(anomalies))

display(
    anomalies[
        [
            "title",
            "popularity",
            "vote_count",
            "popularity_zscore",
            "vote_count_zscore"
        ]
    ].head(20)
)

Total anomaly movies: 26


,title,popularity,vote_count,popularity_zscore,vote_count_zscore
2345,Spider-Man: No Way Home,438.4757,22429,23.456677,2.443120
2495,Disclosure Day,394.3346,2328,21.019194,-1.236290
2497,Obsession,364.8722,4203,19.392273,-0.276869
2479,Backrooms,258.4743,2483,13.516953,-1.131622
195,Spider-Man,180.5936,20991,9.216361,2.335484
1847,Spider-Man: Homecoming,169.0857,23456,8.580892,2.515848
2369,Project Hail Mary,162.7023,6750,8.228399,0.492578
2447,Michael,108.7248,3846,5.247747,-0.421031
2086,Spider-Man: Far From Home,105.6644,17240,5.078751,2.015701
2444,Mortal Kombat II,99.5231,2260,4.739627,-1.284427
